# Photonic PDK validation

Open photonic PDKs ship two things for the same device: a **parametric
geometry cell** driven by design parameters (`gap`, `length`, `radius`,
`width`) and a **foundry-validated compact model** (SAX S-parameters, a
coupling-coefficient lookup table, or a measured spectrum). This notebook
verifies that both are available for the FDTD validation cases and shows the
geometry and the reference data for each.

| Case | Geometry (`gdsfactory`) | Reference data | Source |
| --- | --- | --- | --- |
| 1. Straight waveguide | `ubcpdk.cells.straight` | SAX `waveguides.straight` | UBC SiEPIC EBeam |
| 2. Directional coupler | `ubcpdk.cells.coupler` | κ LUT + measured `opics_ebeam` | UBC / Cornerstone / SiEPIC |
| 3. Ring coupling area | `ubcpdk.cells.coupler_ring` | κ LUT | UBC / Cornerstone |
| 4. MZI spectrum | `ubcpdk.cells.mzi` | measured spectrum + MZI model | UBC SiEPIC EBeam |
| 5. Ring spectrum | `ubcpdk.cells.ring_double` | measured spectrum | UBC SiEPIC EBeam |
| 6. Bragg spectrum | custom `sinusoidal_dbr` | measured spectrum | UBC SiEPIC EBeam |

Compact-model comparisons use the same nominal parameters. Measured-spectrum
cases have no supplied fabricated mask in this dataset, so nominal parameters
do not establish exact geometry matching. The Meep runs compare both 3-D and 2-D effective-index
FDTD for the devices that fit the same reference data.

Measured data is fetched from pinned git revisions by
`tests.pdk_validation._fetch` (neither the UBC `.mat` spectra nor the SiEPIC
coupler `.dat` files ship in the published wheels), and the Cornerstone LUTs
are read directly with xarray by `tests.pdk_validation._cspdk`.

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "pyproject.toml").is_file() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import matplotlib.pyplot as plt
import numpy as np
import gdsfactory as gf
import ubcpdk

ubcpdk.PDK.activate()

from ubcpdk.models import couplers as dc_models
from ubcpdk.models import waveguides as wg_models

from tests.pdk_validation import _cspdk, _fetch

WAVELENGTHS_UM = np.linspace(1.50, 1.60, 201)
print("active PDK:", ubcpdk.PDK.name)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from tests.pdk_validation import _validation as V

# Point GSIM_MEEP_PYTHON at a Python with `pymeep` installed (e.g. a conda env).
MEEP_PYTHON = os.environ.get("GSIM_MEEP_PYTHON", "python")
# Optional directory for the run artifacts (None -> a temporary directory).
OUTPUT_ROOT = os.environ.get("GSIM_PDK_OUTPUT") or None

# Accuracy-first per-case settings: finer mesh, a denser frequency grid and a
# tighter field-energy decay stop than the harness default, so every example is
# pushed as close to its reference as a single workstation allows.
OVERRIDES = V.high_resolution_overrides()


def _db(values):
    return 20 * np.log10(np.clip(np.abs(values), 1e-9, None))


def plot_validation(outcome, title):
    """Overlay the de-embedded Meep S-parameters on the reference target."""
    figure, axis = plt.subplots(figsize=(8, 3.5))
    measured = outcome.reference.transmission_db is not None
    keys = (
        outcome.reference.sim_ports
        or tuple(outcome.reference.s_params)
        or tuple(outcome.s_params)
    )
    for name in keys:
        if name not in outcome.s_params:
            continue
        values = outcome.s_params[name]
        curve = _db(values)
        if measured:  # fiber-referenced data: compare shapes, not absolute loss
            curve = curve - np.nanmax(curve)
        axis.plot(
            outcome.wavelengths,
            curve,
            "-",
            label=f"Meep {outcome.dimension.upper()} {name}",
        )
    if outcome.reference.s_params:
        for name, values in sorted(outcome.reference.s_params.items()):
            axis.plot(
                outcome.reference.wavelengths, _db(values), "--", label=f"ref {name}"
            )
    elif outcome.reference.transmission_db is not None:
        ref_wl, ref_db = outcome.reference.transmission_db
        axis.plot(
            ref_wl, V.normalize_measured_spectrum(ref_db), "k", lw=0.8, label="measured"
        )
    axis.set(title=title, xlabel="wavelength (um)", ylabel="|S| (dB)")
    axis.legend(fontsize=8)
    figure.tight_layout()
    return figure


def run_and_report(name, title, *, dimension=None):
    """Run one validation case, plot the overlay and print agreement + budget."""
    outcome = V.run_case(
        name,
        output_dir=(str(Path(OUTPUT_ROOT) / name) if OUTPUT_ROOT else None),
        python_executable=MEEP_PYTHON,
        verbose=False,
        dimension=dimension,
        **OVERRIDES.get(name, {}),
    )
    plot_validation(outcome, title)
    plt.show()
    for port, metrics in outcome.metrics.items():
        if metrics:
            print(
                f"{port}: RMS={metrics['rms_db']:.2f} dB  "
                f"max={metrics['max_abs_db']:.2f} dB  "
                f"corr={metrics['corr']:.3f}  (n={metrics['n']})"
                if metrics.get("corr") is not None
                else f"corr=n/a  (n={metrics['n']})"
            )
    row = outcome.summary()
    size = (
        f"{row['cell_x_lambda']:.1f} x {row['cell_y_lambda']:.1f} x "
        f"{row['cell_z_lambda']:.1f} lambda"
    )
    print(
        f"{name}: {row['dimension']} res={row['resolution']} "
        f"freqs={row['num_freqs']} | cell {size} | "
        f"area {row['area_lambda2']:.0f} lambda^2 | "
        f"{row['cells'] / 1e6:.2f}M cells | compute {row['wall_seconds']:.0f} s "
        f"({row['timesteps']} steps)"
    )
    return outcome

## Validation methodology

Compact-model cases use the same nominal `gap`/`length`/`radius` as their
reference. The measured ring and Bragg devices have no reference GDS here;
exact geometry matching is unverified. The SiEPIC half-ring measurement is
compared with generic-PDK geometry, so it is a cross-PDK comparison. The
published rerun uses the **2-D effective-index plane** for every case so the
complete UBC + Cornerstone + generic-PDK SiEPIC set is reproducible on one
workstation. The 2-D cut is pinned to the core centre and the background is the
PDK cladding (not the substrate).

| Setting | Value | Why |
| --- | --- | --- |
| `resolution` | 30-40 px/um | Three cells at 30 px/µm or four at 40 px/µm across 100 nm gaps; this is not a mesh-convergence result. The measured MZI is runtime-excluded and is not part of the quantitative table. |
| `num_freqs` | 101 | The measured ring through dips are about 0.17–0.25 nm wide, much narrower than the roughly 1 nm wavelength sampling; the ring spectrum is under-resolved. |
| stopping | field-energy decay | `stop_when_energy_decayed` with a wall-clock safety cap; each run records its actual timesteps. |
| subpixel | off | Disabled in the recorded runs; a subpixel and mesh sensitivity check is still needed, especially for the Bragg corrugation. |
| de-embedding | mode `n_eff` | Moves each monitor back to the port reference plane. |
| provenance | recorded | Each `sim_config.json` records PDK name, overlay, stack hash, resolved layers/materials and mesh settings. |


## Reference data inventory

Both Cornerstone (`cspdk`) and UBC (`ubcpdk`) publish the same strip
directional-coupler and racetrack coupling-coefficient LUTs as xarray tables.
`_cspdk` opens the Cornerstone LUTs directly (the `cspdk` package itself pins a
different gdsfactory release), so the data is available without importing it.

In [ ]:
for model in (_cspdk.DIRECTIONAL_COUPLER_STRIP, _cspdk.RACETRACK_STRIP):
    lut = _cspdk.load_lut(model=model)
    print(f"cspdk {model}: shape={lut.shape}")
    for dim in lut.dims:
        print(
            f"    {dim:>10}: {float(lut[dim].min()):g} .. {float(lut[dim].max()):g} um"
        )

for name in ("directional_coupler_strip.nc", "coupler_racetrack_strip.nc"):
    path = Path(ubcpdk.__file__).parent / "models" / name
    with __import__("xarray").open_dataarray(path) as array:
        print(f"ubcpdk {name}: shape={array.shape}")

## Case 1 — Straight waveguide

**Geometry** (`ubcpdk.cells.straight`) and **reference model**
(`ubcpdk.models.waveguides.straight`, SAX `straight` with `neff=2.38`,
`ng=4.30`, `loss_dB_cm=3`). The model returns `S(o1,o2)`; its phase slope
gives the group index and its magnitude gives the loss — the quantities the
FDTD run must reproduce.

In [ ]:
straight = ubcpdk.cells.straight(length=20.0)
print(
    "ports:",
    [port.name for port in straight.ports],
    "| size:",
    straight.xsize,
    straight.ysize,
    "um",
)
straight

In [ ]:
LENGTH_UM = 100.0
s21 = np.asarray(wg_models.straight(wl=WAVELENGTHS_UM, length=LENGTH_UM)[("o1", "o2")])
phase = np.unwrap(np.angle(s21))
ng = -(WAVELENGTHS_UM**2) * np.gradient(phase, WAVELENGTHS_UM) / (2 * np.pi * LENGTH_UM)
loss_db = -20 * np.log10(np.abs(s21))

figure, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(WAVELENGTHS_UM, ng)
axes[0].set(title="Group index", xlabel="wavelength (um)", ylabel="ng")
axes[1].plot(WAVELENGTHS_UM, loss_db)
axes[1].set(title="Propagation loss", xlabel="wavelength (um)", ylabel="loss (dB)")
figure.tight_layout()

In [ ]:
straight_meep = run_and_report("straight", "Meep vs SAX |S| — straight")

## Case 2 — Directional coupler

**Geometry** (`ubcpdk.cells.coupler`) and **reference model**
(`ubcpdk.models.couplers.directional_coupler`), which evaluates the κ LUT over
`wavelength`, `gap`, `radius`, `length_x`, `v_offset` and returns the
through (`tau`) and cross (`j*kappa`) S-parameters at ports `o1..o4`.

In [ ]:
coupler = ubcpdk.cells.coupler(gap=0.2, length=10.0)
print(
    "ports:",
    [port.name for port in coupler.ports],
    "| size:",
    coupler.xsize,
    coupler.ysize,
    "um",
)
coupler

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for gap in (0.15, 0.2, 0.3):
    model = dc_models.directional_coupler(wl=WAVELENGTHS_UM, length=10.0, gap=gap)
    kappa = np.abs(np.asarray(model[("o1", "o3")]))
    axes[0].plot(WAVELENGTHS_UM, kappa, label=f"gap={gap} um")
axes[0].set(title="Coupling vs wavelength", xlabel="wavelength (um)", ylabel="|kappa|")
axes[0].legend()

gaps = np.linspace(0.05, 0.5, 40)
kappa_1550 = np.array(
    [
        float(
            np.abs(
                np.asarray(
                    dc_models.directional_coupler(wl=1.55, length=10.0, gap=gap)[
                        ("o1", "o3")
                    ]
                )
            )
        )
        for gap in gaps
    ]
)
axes[1].plot(gaps, kappa_1550, "o-")
axes[1].set(title="Coupling at 1.55 um", xlabel="gap (um)", ylabel="|kappa|")
figure.tight_layout()

**Measured reference (SiEPIC `opics_ebeam`).** The measured coupler
S-parameters live in `SiEPIC/SiEPIC_EBeam_PDK` (fetched via `_fetch`); each
block holds `(frequency, |S|, phase)`. The `o1`-excited responses are shown
below.

In [ ]:
try:
    coupler_dat = _fetch.fetch_siepic_coupler(gap_nm=100, radius_um=10)
    blocks = _fetch.load_siepic_coupler(coupler_dat)
    figure, axis = plt.subplots(figsize=(8, 3.5))
    for (receive, source), values in sorted(blocks.items()):
        if source != 1:
            continue
        wavelength_nm = 299792458.0 / values[:, 0] * 1e9
        axis.plot(wavelength_nm, values[:, 1], label=f"|S({receive},{source})|")
    axis.set(
        title="Measured SiEPIC coupler (o1 excited)",
        xlabel="wavelength (nm)",
        ylabel="|S|",
    )
    axis.legend()
except _fetch.FetchError as error:
    print("Measured coupler unavailable:", error)

In [ ]:
coupler_meep = run_and_report("coupler", "Meep vs SAX LUT — directional coupler")

## Case 3 — Ring/racetrack coupling area

**Geometry** (`ubcpdk.cells.coupler_ring`) and **reference model**
(`ubcpdk.models.couplers.coupler_ring_coupling_area`), which evaluates the
racetrack κ LUT over `wavelength`, `gap`, `radius`, `length_x`.

In [ ]:
gf.clear_cache()
ring = ubcpdk.cells.coupler_ring(length_x=4.0, radius=10.0, gap=0.1)
print(
    "ports:",
    [port.name for port in ring.ports],
    "| size:",
    ring.xsize,
    ring.ysize,
    "um",
)
ring

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for radius in (5.0, 10.0, 20.0):
    model = dc_models.coupler_ring_coupling_area(
        wl=WAVELENGTHS_UM, gap=0.1, radius=radius, length_x=2.0
    )
    kappa = np.abs(np.asarray(model[("o1", "o3")]))
    axes[0].plot(WAVELENGTHS_UM, kappa, label=f"radius={radius} um")
axes[0].set(
    title="Racetrack coupling vs wavelength", xlabel="wavelength (um)", ylabel="|kappa|"
)
axes[0].legend()

gaps = np.linspace(0.05, 0.5, 40)
kappa_1550 = np.array(
    [
        float(
            np.abs(
                np.asarray(
                    dc_models.coupler_ring_coupling_area(
                        wl=1.55, gap=gap, radius=10.0, length_x=2.0
                    )[("o1", "o3")]
                )
            )
        )
        for gap in gaps
    ]
)
axes[1].plot(gaps, kappa_1550, "o-")
axes[1].set(title="Racetrack coupling at 1.55 um", xlabel="gap (um)", ylabel="|kappa|")
figure.tight_layout()

In [ ]:
ring_coupling_meep = run_and_report(
    "ring_coupling", "Meep vs SAX LUT — ring coupling area"
)

## Case 4 — MZI spectrum

**Geometry** (`ubcpdk.cells.mzi`) with the **measured** spectrum as the
validation target, alongside the compact MZI model
(`ubcpdk.simulation.circuits.mzi_spectrum`).

In [ ]:
mzi = ubcpdk.cells.mzi(delta_length=10.0)
print(
    "ports:", [port.name for port in mzi.ports], "| size:", mzi.xsize, mzi.ysize, "um"
)
mzi

In [ ]:
from ubcpdk.simulation.circuits.mzi_spectrum import mzi_spectrum
from ubcpdk.simulation.circuits.waveguide import wavelength_um as mzi_wavelength_um

figure, axis = plt.subplots(figsize=(8, 3.5))
try:
    mzi_dat = _fetch.fetch_ubc(_fetch.UBC_MZI_MAT)
    wavelength_nm, power_dbm = _fetch.load_ubc_spectrum(mzi_dat)
    axis.plot(wavelength_nm, power_dbm, "k", lw=0.6, label="measured")
except _fetch.FetchError as error:
    print("Measured MZI unavailable:", error)

model = mzi_spectrum(L1_um=40, L2_um=255)
axis.plot(mzi_wavelength_um * 1000, 10 * np.log10(model) - 20, "--", label="MZI model")
axis.set(title="MZI transmission", xlabel="wavelength (nm)", ylabel="power (dB)")
axis.legend()

In [ ]:
print(
    "MZI excluded from quantitative validation: runtime too large at >=20 px/um; see runs/_excluded_runtime/README.md"
)

## Case 5 — Ring resonator spectrum

**Geometry** (`ubcpdk.cells.ring_double`) with the **measured** double-ring
spectrum (radius 10 µm, 100 nm gap) as a nominal-parameter comparison. The fabricated mask is not supplied.

In [ ]:
gf.clear_cache()
ring = ubcpdk.cells.ring_double(radius=10.0, gap=0.1)
print(
    "ports:",
    [port.name for port in ring.ports],
    "| size:",
    ring.xsize,
    ring.ysize,
    "um",
)
ring

In [ ]:
figure, axis = plt.subplots(figsize=(8, 3.5))
try:
    ring_dat = _fetch.fetch_ubc(_fetch.UBC_RING_MAT)
    wavelength_nm, power_dbm = _fetch.load_ubc_spectrum(ring_dat, port=1)
    axis.plot(wavelength_nm, power_dbm, "k", lw=0.6)
    axis.set(
        title="Measured ring inferred through detector 1 (TE r10 g100)",
        xlabel="wavelength (nm)",
        ylabel="power (dBm)",
    )
except _fetch.FetchError as error:
    print("Measured ring unavailable:", error)

In [ ]:
ring_meep = run_and_report("ring", "Meep vs measured — ring resonator")

## Case 6 — Bragg grating spectrum

**Simulated geometry:** the 200-period, 120 nm-modulation sinusoidal polygon
shown below. The pinned UBC Set 1 design #10 specifies these nominal
parameters, but the measured detector power includes a reflection fixture
with a 2×2 splitter and grating couplers. Detector column 1 is the inferred
reflection channel; its mapping and fixture transfer function are not
independently verified. This is a diagnostic overlay, not a direct S11
validation.


In [ ]:
# Display the same component factory used by the recorded Meep case.
bragg = V.sinusoidal_dbr(
    n_periods=V.DBR_PERIODS,
    period=V.DBR_PERIOD_UM,
    width=V.DBR_W1,
    delta_width=V.DBR_DELTA_WIDTH_UM,
)
print(
    "ports:", [p.name for p in bragg.ports], "| size:", bragg.xsize, bragg.ysize, "um"
)
bragg

In [ ]:
figure, axis = plt.subplots(figsize=(8, 3.5))
try:
    bragg_dat = _fetch.fetch_ubc(_fetch.UBC_BRAGG_MAT)
    wavelength_nm, power_dbm = _fetch.load_ubc_spectrum(bragg_dat, port=1)
    axis.plot(wavelength_nm, power_dbm, "k", lw=0.6)
    axis.set(
        title="Bragg detector column 1 (fixture response)",
        xlabel="wavelength (nm)",
        ylabel="power (dBm)",
    )
except _fetch.FetchError as error:
    print("Measured Bragg unavailable:", error)

In [ ]:
bragg_meep = run_and_report("bragg", "Meep vs measured — Bragg grating")

## Additional PDK targets — Cornerstone and generic PDK

The validation set now covers the two primary PDK data sources plus the measured
SiEPIC coupler:

* **UBC (`ubcpdk`)** — the six original measured/model cases.
* **Cornerstone (`cspdk.si220`)** — directional-coupler and racetrack coupling
  cases against the Cornerstone LUTs, using the Cornerstone LayerStack and
  derived `core = WG - GRA` geometry.
* **SiEPIC measured coupler** — geometry from the generic photonic PDK
  (`gf.gpdk`, 220 nm Si), compared with the measured SiEPIC half-ring data.

Each target activates its own PDK (and clears gdsfactory's cross-PDK cell
cache) before building the matched device. Recorded runs keep the PDK name,
overlay, stack hash, and resolved material table for audit.


In [ ]:
for name in ("cspdk_coupler", "cspdk_ring_coupling", "siepic_coupler"):
    try:
        outcome = run_and_report(name, f"Meep vs reference — {name}")
        plt.show()
    except KeyError:
        print(f"{name}: target PDK not installed in this kernel")

## Diagnostic reports

These reports make the PDK interpretation auditable: resolved layers and roles,
material properties at 1.55 µm, drawn geometry, feature-vs-grid resolution, and
the provenance recorded in each `sim_config.json`.


In [ ]:
from gsim.common.pdk import resolve_layer_stack
from gsim.common.pdk.diagnostics import (
    feature_resolution,
    geometry_report,
    material_report,
    stack_report,
)
from tests.pdk_validation._pdks import available_targets

targets = available_targets()
for key in ("ubc", "cspdk", "generic"):
    if key not in targets:
        print(f"{key}: target not installed")
        continue
    target = targets[key]
    target.activate()
    case = next(c for c in V.build_cases() if c.target.key == key)
    component = case.make_component()
    resolved = resolve_layer_stack(
        component, pdk=target.pdk, dielectrics=target.dielectrics, wavelength_um=1.55
    )
    print(f"\n### {key}: {case.name} ({resolved.pdk_name}, overlay={resolved.overlay})")
    print(
        "layers:",
        [
            (row["layer"], row["layer_type"], row["mesh_order"], row["drawn"])
            for row in stack_report(resolved)
        ],
    )
    materials = material_report(resolved, wavelength_um=1.55)
    print(
        "materials:",
        {
            name: (data.get("refractive_index"), data.get("behavior"))
            for name, data in materials.items()
        },
    )
    geometry = geometry_report(component, resolved, wavelength_um=1.55)
    print(
        "drawn area (um^2):",
        {
            row["layer"]: row["area_um2"]
            for row in geometry["layers"]
            if row["area_um2"]
        },
    )
    print("ports:", [(port["name"], port["layer"]) for port in geometry["ports"]])
    print(
        "feature resolution:",
        V.case_feature_report(case.name, resolution=case.resolution),
    )

### Recorded provenance

Each committed run records the PDK name, overlay, stack hash and resolved layer/material counts.


In [ ]:
import json
from pathlib import Path

runs = Path("nbs/data/pdk_validation/runs")
for case in V.build_cases():
    run = V.find_recorded_run(case.name, runs, dimension="2d", fallback_to_2d=False)
    if run is None:
        continue
    cfg = json.loads((run / "sim_config.json").read_text())
    print(
        case.name,
        "pdk=",
        cfg.get("pdk_name"),
        "overlay=",
        cfg.get("pdk_overlay"),
        "hash=",
        str(cfg.get("pdk_stack_hash"))[:12],
        "layers=",
        len(cfg.get("layer_stack", [])),
        "materials=",
        len(cfg.get("materials", {})),
    )

## Summary

The compact-model cases use matched nominal parameters and were run with
feature-resolved meshes, `num_freqs=101`, field-energy decay stopping, monitor
de-embedding to the port reference planes, and insertion-loss normalisation for
the measured spectra. `run_and_report` returns a `ValidationOutcome` whose
`.metrics` holds RMS / max dB error and correlation per port.

Reference targets: straight -> SAX model; coupler and ring coupling area ->
SAX/Cornerstone coupling LUTs at nominally corresponding geometry; MZI, ring and Bragg ->
measured UBC spectra (normalised).

The run cells below compare the 3-D-feasible devices with paired 2-D
effective-index results, using the same compact-model or measured reference
for each geometry. The MZI and the large ring/coupling cases use the 2-D effective-index plane
because their 3-D grids are too large for a practical full-spectrum run. UBC and
Cornerstone use their authoritative LayerStacks; the SiEPIC measured coupler is
built with the generic photonic PDK (220 nm Si).


In [ ]:
# Run every case in one go (long; needs GSIM_MEEP_PYTHON). Results are also
# returned as a dict for programmatic post-processing.
outcomes = V.run_all(
    names=[case.name for case in V.build_cases() if case.name != "mzi"],
    python_executable=MEEP_PYTHON,
    verbose=False,
    overrides=OVERRIDES,
)
for name, outcome in outcomes.items():
    if isinstance(outcome, Exception):
        print(f"{name}: FAILED ({outcome})")
    else:
        summary = {
            port: round(m["rms_db"], 2) for port, m in outcome.metrics.items() if m
        }
        print(f"{name}: RMS dB per port {summary}")

In [ ]:
# Pair 2-D effective-index runs with the existing 3-D outcomes. MZI is already
# a 2-D case because its 3-D grid is prohibitively large.
two_d_outcomes = {}
for name in [
    case.name
    for case in V.build_cases()
    if case.dimension == "2d" and case.name != "mzi"
]:
    outcome = V.run_case(
        name,
        output_dir=(str(Path(OUTPUT_ROOT) / f"{name}-2d") if OUTPUT_ROOT else None),
        python_executable=MEEP_PYTHON,
        verbose=False,
        dimension="2d",
        **OVERRIDES.get(name, {}),
    )
    two_d_outcomes[name] = outcome
    plot_validation(outcome, f"Meep 2-D effective index vs reference — {name}")
    plt.show()
    for port, metrics in outcome.metrics.items():
        if metrics:
            print(
                f"{name} {port}: RMS={metrics['rms_db']:.2f} dB  "
                f"max={metrics['max_abs_db']:.2f} dB  corr={metrics['corr']:.3f}"
                if metrics.get("corr") is not None
                else f"max={metrics['max_abs_db']:.2f} dB  corr=n/a"
            )
    print("2-D run:", outcome.summary())

display(Markdown(V.format_summary(two_d_outcomes, title="2-D effective-index runs")))

## Simulation plan and recorded run status

The run cells below compare nominal-parameter PDK geometries against SAX/Cornerstone compact models or measured spectra. Exact geometry matching for measured cases is unverified. The recorded results are the fresh authoritative-stack runs committed under `data/pdk_validation/runs/`; seven comparisons and one straight-guide pipeline diagnostic were re-run locally. The measured MZI is runtime-excluded and is not counted as a validation result.

Measured spectra are normalized to their 99th-percentile level before computing dB errors. Compact-model S-parameters are compared at absolute level. `RMS` and `max` are dB errors; `bias` is mean simulation-minus-reference error. Correlation is useful for spectrally varying curves but not for nearly flat straight-waveguide transmission.


In [ ]:
from IPython.display import Markdown, display

display(
    Markdown(V.format_summary(outcomes, title="Compute time and size in wavelengths"))
)

# Flat rows for downstream use (one dict per example).
budget_rows = [
    outcome.summary()
    for outcome in outcomes.values()
    if isinstance(outcome, V.ValidationOutcome)
]
budget_rows

## Recorded validation results (fresh authoritative-stack rerun)

These are the actual committed results, not an earlier narrative. The publication rerun uses the 2-D effective-index plane. The measured MZI is **runtime-excluded**: it is too large to run at >=20 px/um in the validation window on this workstation, and its coarse smoke run is kept under `runs/_excluded_runtime/` rather than being cited as validation.

| Case | res | best port | RMS (dB) | corr |
| --- | --- | --- | --- | --- |
| straight | 30 | S21 | 0.01 | -0.21347151767975023 |
| coupler | 30 | S41 | 4.43 | 0.9977595206473594 |
| ring_coupling | 40 | S41 | 9.47 | -0.9890151543430252 |
| ring | 40 | S21 | 8.00 | -0.0007581032685122574 |
| bragg | 30 | S11 | 23.37 | 0.2312094182647101 |
| cspdk_coupler | 30 | S41 | 7.25 | 0.999814546140312 |
| cspdk_ring_coupling | 40 | S41 | 1.91 | -0.9827298481267679 |
| siepic_coupler | 40 | S31 | 1.20 | -0.9904786927076473 |

The straight guide gives |S21| ≈ 1 with 0.006 dB RMS error. Both the SAX reference and lossless 2-D Meep model predict nearly flat unit transmission, so this is a pipeline check and does not discriminate process or geometry accuracy. The UBC coupler tracks the wavelength trend, but S41 has a −4.4 dB bias while S31 has a +4.7 dB bias: this is a splitting-ratio error, not a common level offset. Several ring comparisons are anti-correlated despite finite RMS values. The ring and Bragg cases remain materially discrepant. In particular, the pinned UBC Bragg design uses a 2×2 splitter and grating-coupler reflection fixture; the measured detector power is normalized but not fixture-de-embedded. Its 23.37 dB RMS comparison with bare-grating S11 is diagnostic only. Raw spectra, configs, diagnostics and debug metadata are under `data/pdk_validation/runs/`.


## Recorded UBC coupler mesh sensitivity

The same GDS, port definitions and asymmetric-slab core permittivity were used at
30, 50 and 70 pixels/µm. The refined recorded spectra are loaded below through
the same de-embedding and LUT comparison code as the main case. The table and
curves show a modest improvement against the LUT. They do **not** establish
complex-S convergence: the 50-to-70 changes exceed 0.01, the 50-pixel run used
a tighter decay threshold, and the 70-pixel run stopped at about 200 Meep time
units while the other two reached about 250. Complex changes below are
de-embedded to the device port planes; the audit also reports raw monitor-plane
changes. All three spectra are archived in
`data/pdk_validation/mesh_sweeps/` and `data/pdk_validation/runs/2d_coupler/`.


In [ ]:
import json

repo_root = next(
    candidate
    for candidate in (Path.cwd(), Path.cwd().parent)
    if (candidate / "pyproject.toml").is_file()
)
mesh_data = repo_root / "nbs" / "data" / "pdk_validation"
mesh_paths = {
    30: mesh_data / "runs" / "2d_coupler",
    50: mesh_data / "mesh_sweeps" / "coupler_r50",
    70: mesh_data / "mesh_sweeps" / "coupler_r70",
}
mesh_outcomes = {
    res: V.load_outcome("coupler", path) for res, path in mesh_paths.items()
}
mesh_summary = {
    30: json.loads((mesh_data / "runs" / "summary.json").read_text())["coupler"][
        "summary"
    ],
    50: json.loads((mesh_paths[50] / "summary.json").read_text())["coupler"]["summary"],
    70: json.loads((mesh_paths[70] / "summary.json").read_text())["coupler"]["summary"],
}
assert all(
    np.allclose(mesh_outcomes[30].wavelengths, mesh_outcomes[res].wavelengths)
    for res in (50, 70)
)

rows = [
    "| pixels/µm | cells | Meep time | S41 RMS (dB) | S31 RMS (dB) |",
    "| ---: | ---: | ---: | ---: | ---: |",
]
for res, outcome in mesh_outcomes.items():
    meta = mesh_summary[res]
    rows.append(
        f"| {res} | {meta['cells']:,} | {meta['meep_time']:.0f} | "
        f"{outcome.metrics['S41']['rms_db']:.2f} | {outcome.metrics['S31']['rms_db']:.2f} |"
    )
display(Markdown("\n".join(rows)))

figure, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for axis, port in zip(axes, ("S41", "S31")):
    for res, outcome in mesh_outcomes.items():
        axis.plot(
            outcome.wavelengths, _db(outcome.s_params[port]), label=f"Meep {res} px/µm"
        )
    reference = mesh_outcomes[30].reference
    axis.plot(
        reference.wavelengths, _db(reference.s_params[port]), "k--", label="UBC LUT"
    )
    axis.set(title=port, xlabel="wavelength (µm)", ylabel="magnitude (dB)")
    axis.grid(alpha=0.25)
axes[0].legend(fontsize=8)
figure.tight_layout()
plt.show()

for old, new in ((30, 50), (50, 70)):
    values = [
        np.sqrt(
            np.mean(
                np.abs(
                    mesh_outcomes[new].s_params[port]
                    - mesh_outcomes[old].s_params[port]
                )
                ** 2
            )
        )
        for port in ("S41", "S31")
    ]
    print(
        f"{old}→{new} de-embedded complex RMS change: S41={values[0]:.3f}, S31={values[1]:.3f}"
    )